# Hipótese 2: Alarme em portas + usadas(extremos ou meio)

## 1. Contexto e Fundamentação Teórica
As Portas de Plataforma (PSDs) são equipamentos de segurança distribuídos ao longo das plataformas e submetidos continuamente à interação com os passageiros durante o embarque e o desembarque. Embora todas realizem ciclos de abertura e fechamento, sua localização pode gerar diferentes níveis de exposição ao fluxo de pessoas. Portas próximas a escadas, escadas rolantes, elevadores e acessos à plataforma tendem a receber maior concentração de passageiros do que aquelas situadas em regiões mais afastadas.

A maior interação com passageiros pode aumentar tanto a ocorrência de obstáculos durante o movimento das folhas quanto o desgaste dos componentes mecânicos e eletrônicos das portas. Consequentemente, portas mais utilizadas podem apresentar maior frequência de diferentes alarmes e defeitos, como obstáculos (`OBS`), esforço excessivo, tempos excedidos de abertura ou fechamento e falhas relacionadas aos sensores.

A hipótese operacional deste estudo estabelece, portanto, que a posição da porta na plataforma está associada à frequência de alarmes e defeitos, com expectativa de maior incidência nas portas mais utilizadas, geralmente próximas ao centro ou aos acessos em comparação às portas menos utilizadas, localizadas nas extremidades. A análise contemplará os diferentes tipos de alarme vinculados a cada porta. Especificamente para os registros `OBS`, que representam obstáculos, serão consideradas apenas ocorrências com duração superior a 60 segundos; registros `OBS` com duração igual ou inferior a 60 segundos, ausente ou inválida serão desconsiderados. Esse filtro não será aplicado aos demais tipos de alarme.

Como a base não contém medições diretas do fluxo de passageiros nem a localização dos acessos de cada estação, a posição geométrica da porta será utilizada como aproximação de uso. Dessa forma, os resultados permitirão identificar associações entre posição e frequência de alarmes, mas não comprovar, isoladamente, que o fluxo de passageiros é a causa direta dos defeitos encontrados.

---

## 2. Importação das bibliotecas e auditoria inicial

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from IPython.display import display

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 120

alarmes_path = "../../data/Alarmes do sistema de PSD.xlsx"
df_alarmes = pd.read_excel(alarmes_path)

print(f"Dimensão da base: {df_alarmes.shape[0]:,} linhas × {df_alarmes.shape[1]} colunas")
display(df_alarmes.head())

Dimensão da base: 150,003 linhas × 7 colunas


,TAG,Horário do Alarme,Estação,Porta,Alarme,Horário da Normalizacao,Tempo de Normalização
0,PSD_FAL_FAL2_PDM09_OBS,2026-05-12 08:16:52.657,FAL,9.0,OBS,2026-05-12 08:16:53.670,"0 dias, 00:00:01"
1,PSD_LUZ_LUZ1_PDM08_OBS,2026-05-12 08:15:09.913,LUZ,8.0,OBS,2026-05-12 08:15:10.917,"0 dias, 00:00:01"
2,PSD_FAL_FAL1_PDM15_OBS,2026-05-12 08:14:06.657,FAL,15.0,OBS,2026-05-12 08:14:07.670,"0 dias, 00:00:01"
3,PSD_LUZ_LUZ1_PDM19_OBS,2026-05-12 08:09:31.930,LUZ,19.0,OBS,2026-05-12 08:09:32.917,"0 dias, 00:00:01"
4,PSD_FRE_FRE1_PDM14_OBS,2026-05-12 08:05:05.830,FRE,14.0,OBS,2026-05-12 08:05:14.847,"0 dias, 00:00:09"


In [5]:
# Auditoria inicial da base, sem alterar ou excluir registros
resumo_colunas = pd.DataFrame({
    "tipo": df_alarmes.dtypes.astype(str),
    "valores_ausentes": df_alarmes.isna().sum(),
    "percentual_ausente": (df_alarmes.isna().mean() * 100).round(2),
    "valores_unicos": df_alarmes.nunique(dropna=True),
})

print(f"Duplicatas exatas: {df_alarmes.duplicated().sum():,}")
display(resumo_colunas)

Duplicatas exatas: 0


,tipo,valores_ausentes,percentual_ausente,valores_unicos
TAG,str,1,0.00,379
Horário do Alarme,datetime64[us],37252,24.83,36477
Estação,str,3,0.00,6
Porta,float64,3,0.00,24
Alarme,str,3,0.00,8
Horário da Normalizacao,datetime64[us],3,0.00,35360
Tempo de Normalização,str,3,0.00,56673


## 3. Pré-processamento e Limpeza dos Dados

A base não possui duplicatas exatas, mas contém registros sem horário de alarme e três linhas sem identificação de estação, porta e tipo de alarme. Nesta etapa, os dados serão tratados sem modificar a base original. As regras de duração serão aplicadas apenas aos alarmes de obstáculo (`OBS`); os demais tipos de alarme serão preservados para a análise.

In [6]:
# Cópia de trabalho para preservar a base original
df = df_alarmes.copy()

colunas_identificacao = ["Estação", "Porta", "Alarme"]
linhas_sem_identificacao = df[colunas_identificacao].isna().any(axis=1)

print(f"Linhas sem identificação removidas: {linhas_sem_identificacao.sum():,}")
display(df.loc[linhas_sem_identificacao])

df = df.loc[~linhas_sem_identificacao].copy().reset_index(drop=True)

print(f"Dimensão após a remoção: {df.shape[0]:,} linhas × {df.shape[1]} colunas")

Linhas sem identificação removidas: 3


,TAG,Horário do Alarme,Estação,Porta,Alarme,Horário da Normalizacao,Tempo de Normalização
150000,NaN,NaT,NaN,NaN,NaN,NaT,NaN
150001,Nenhum filtro aplicado,NaT,NaN,NaN,NaN,NaT,NaN
150002,Exported data exceeded the allowed volume. Som...,NaT,NaN,NaN,NaN,NaT,NaN


Dimensão após a remoção: 150,000 linhas × 7 colunas
